# Used-Car Price Prediction - end-to-end on Google Colab
Run **Runtime ▸ Run all**. Everything (raw + cleaned data, figures, models, MLflow runs, reports, logs and the checkpoint manifest) is written to **your Google Drive**.

### If Colab disconnects
Just reconnect and **Run all** again. Every expensive stage - and every individual model experiment - is a checkpoint. On re-run the log prints `SKIP … (verified)` for finished work (the saved files are re-hashed against the manifest first) and `RUN` only for what is missing or changed. To force a stage again put its name in `FORCE_STAGES`.

| step | what | resumable unit |
|---|---|---|
| 1 | download, clean, validate, split | `download`, `clean`, `split` |
| 2 | EDA | `eda` |
| 3 | feature engineering ablation | `ablation` |
| 4 | compare ≥15 regressors | one checkpoint **per model × feature set** |
| 5 | hyper-parameter tuning | one per tuned model (Optuna trials also journalled) |
| 6 | selection + final test evaluation + intervals | `select`, `final` |
| 7 | explainability | `explain` |
| 8 | versioned model bundle + MLflow | `package` |

In [ ]:
#@title ⚙️ Configuration  (edit, then Runtime ▸ Run all)
DATASET = "quikr"                          #@param ["quikr", "cardekho_v3"]
DRIVE_PROJECT_DIR = "car-price-prediction" #@param {type:"string"}
CODE_REPO_URL = "https://github.com/anil-67140/Data-Science-Projects.git" #@param {type:"string"}
CODE_SUBDIR = "campus-x-projects/03-car-price-predictior"                 #@param {type:"string"}
QUICK_MODE = False                         #@param {type:"boolean"}
FORCE_STAGES = []                          # e.g. ["models"] to recompute a stage even if verified
LAUNCH_APP = False                         #@param {type:"boolean"}


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if os.environ.get("CARPRICE_NB_QUICK") == "1":      # local dry-run switch (used by the author's test, harmless in Colab)
    QUICK_MODE = True
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                    # prompts once; persists for the session
    HOME = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
    os.environ["CARPRICE_MLFLOW_DIR"] = "/content/mlruns"   # SQLite is unreliable on the Drive mount; synced to Drive at the end
else:
    HOME = Path(os.environ.get("CARPRICE_HOME", "./car_price_run")).resolve()
HOME.mkdir(parents=True, exist_ok=True)
os.environ["CARPRICE_HOME"] = str(HOME)
print("Project data/artifacts/reports/checkpoints live in:", HOME)

In [ ]:
# --- get the code: use the current checkout if present, otherwise clone it ---
def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "carprice").exists():
            return q
    return None
ROOT = find_root()
if ROOT is None:
    clone = Path("/content/repo") if IN_COLAB else Path("./repo_clone")
    if not clone.exists():
        subprocess.run(["git", "clone", "--depth", "1", CODE_REPO_URL, str(clone)], check=True)
    ROOT = clone / CODE_SUBDIR
    assert (ROOT / "src" / "carprice").exists(), f"{CODE_SUBDIR} does not contain the project - push the upgraded project there first"
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
print("Code root:", ROOT)

In [ ]:
# --- dependencies (Colab already has most; this fills the gaps). Skipped quickly when everything is present. ---
import importlib.util
need = [m for m in ["xgboost","lightgbm","catboost","interpret","shap","optuna","mlflow","pandera","gradio","fastapi","evidently","kaggle"]
        if importlib.util.find_spec(m) is None]
if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    print("all dependencies present")

In [ ]:
# --- Kaggle API credentials: NEVER hard-coded. Order: Colab Secrets -> env vars -> kaggle.json on your Drive ---
def setup_kaggle():
    user = key = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            user, key = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")   # Colab ▸ 🔑 Secrets
        except Exception:
            pass
    user, key = user or os.environ.get("KAGGLE_USERNAME"), key or os.environ.get("KAGGLE_KEY")
    if user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
        return "env/secrets"
    for cand in [HOME / "secrets" / "kaggle.json", Path.home() / ".kaggle" / "kaggle.json"]:
        if cand.exists():
            dest = Path.home() / ".kaggle"; dest.mkdir(exist_ok=True)
            if cand.resolve() != (dest / "kaggle.json").resolve():
                shutil.copy(cand, dest / "kaggle.json")
            (dest / "kaggle.json").chmod(0o600)
            return str(cand)
    return None
from carprice.config import QUICK_OVERRIDES, deep_update, load_settings
overrides = {"dataset": {"active": DATASET}}
if QUICK_MODE:
    overrides = deep_update(overrides, QUICK_OVERRIDES); print("⚡ QUICK_MODE: small model set - for smoke tests only, NOT for reported results")
settings = load_settings(overrides, home=HOME)
if settings.dataset["source"]["type"] == "kaggle":
    where = setup_kaggle()
    assert where, "Kaggle credentials not found. Add KAGGLE_USERNAME / KAGGLE_KEY in Colab Secrets (see README ▸ Kaggle API setup)."
    print("Kaggle credentials found via:", where)
else:
    print(f"Dataset '{DATASET}' is downloaded from {settings.dataset['source']['type']} - no Kaggle credentials needed.")

In [ ]:
import json, warnings, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
warnings.filterwarnings("ignore")
from carprice.pipeline import run_pipeline
P = settings.paths
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", "{:,.3f}".format)
def md(t): display(Markdown(t))
def fig(name): display(Image(filename=str(P.figures / name), width=850))
def status():
    mf = P.manifest
    if not mf.exists():
        print("No checkpoints yet - fresh run."); return
    e = json.load(open(mf))["entries"]
    df = pd.DataFrame([{"unit": k, "status": v["status"], "finished": v.get("finished"), "seconds": v.get("seconds")} for k, v in e.items()])
    display(df.tail(60)); print(f"{(df.status=='complete').sum()} units complete, {(df.status!='complete').sum()} not complete")

## Step 1 · Download, clean, validate, split
⏱ about a minute.

In [ ]:
res = run_pipeline(settings, ["download", "clean", "split"], FORCE_STAGES, mlflow=False)
display(pd.read_csv(P.metrics / "cleaning_report.csv"))
display(pd.read_csv(P.metrics / "outlier_review.csv"))
prov = json.load(open(P.reports / "dataset_sources.json"))[DATASET]
md(f"**Source:** `{prov['acquired_via']}` · sha256 `{prov['sha256'][:16]}…` · rows {prov['raw_rows']}  \n**Licence / terms:** {prov['declared_license'] or 'see Kaggle metadata'}")

## Step 2 · Exploratory data analysis

In [ ]:
run_pipeline(settings, ["eda"], FORCE_STAGES, mlflow=False)
md(open(P.reports / "eda_findings.md").read())
for f in ["eda_01_price_distribution.png", "eda_02_price_by_brand.png", "eda_04_price_vs_year.png", "eda_05_price_vs_mileage.png", "eda_06_price_by_fuel.png"]:
    fig(f)

## Step 3 · Feature engineering ablation
⏱ 1-2 minutes.

In [ ]:
run_pipeline(settings, ["ablation"], FORCE_STAGES, mlflow=False)
ab = json.load(open(P.metrics / "ablation.json"))
display(pd.DataFrame(ab["variant_table"]).pivot(index="variant", columns="model", values="mae_inr"))
print("probe winner:", ab["chosen_variant"], "| average ranks:", ab["average_rank"])

## Step 4 · Compare ≥ 15 regression models
⏱ ~5-10 min on Colab CPU (EBM is the slowest). **Each model is checkpointed - safe to interrupt and re-run.**

In [ ]:
run_pipeline(settings, ["models"], FORCE_STAGES, mlflow=False)
t = pd.read_csv(P.metrics / "model_comparison.csv")
display(t[["model","variant","status","mae_inr","mae_se","rmse_inr","r2","rmsle","mase","fit_s","predict_ms_per_row","model_kb"]])
print(f"{t.model.nunique()} distinct models compared; failures: {int((t.status!='ok').sum())}")
print("excluded:", json.load(open(P.metrics / "excluded_models.json")))

## Step 5 · Hyper-parameter optimisation (Grid / Randomized / Optuna)
⏱ ~5 min.

In [ ]:
run_pipeline(settings, ["tune"], FORCE_STAGES, mlflow=False)
display(pd.read_csv(P.metrics / "tuning_results.csv")[["model","variant","method","n_trials","default_mae_inr","mae_inr","mae_gain_pct","search_seconds","best_params"]])

## Step 6 · Model selection and one-shot test evaluation

In [ ]:
run_pipeline(settings, ["select", "final"], FORCE_STAGES, mlflow=False)
sel = json.load(open(P.metrics / "selection.json")); md("**Selected:** `%s` on `%s` - %s" % (sel["selected"], sel["selected_variant"], sel["rationale"]))
tm = json.load(open(P.metrics / "test_metrics.json"))
display(pd.DataFrame({"selected": tm["test_metrics"], "median baseline": tm["median_baseline_test_metrics"]}).drop(["n"], errors="ignore"))
ir = json.load(open(P.metrics / "interval_report.json"))["conformal"]
md(f"**Prediction interval coverage on held-out data:** {ir['empirical_coverage']:.1%} ± {ir['coverage_se']:.1%} (target {ir['target_coverage']:.0%}); median width ₹{ir['median_width_inr']:,.0f}")
for f in ["test_residual_overview.png", "test_error_by_segment.png", "test_prediction_intervals.png"]:
    fig(f)

## Step 7 · Explainability

In [ ]:
run_pipeline(settings, ["explain"], FORCE_STAGES, mlflow=False)
for f in ["permutation_importance.png", "shap_factor_importance.png", "pdp_ice_year_kms.png"]:
    fig(f)

## Step 8 · Versioned model bundle + MLflow tracking

In [ ]:
run_pipeline(settings, ["package"], FORCE_STAGES, mlflow=True)
md((P.model_reports / "MODEL_CARD.md").read_text())
from carprice.serving.predictor import Predictor, VehicleInput
pred = Predictor.from_dir(models_root=P.models, strict_env=True)      # runs hash + environment + golden-prediction checks
p = pred.predict_one(VehicleInput("Maruti", "Swift", 2015, 45000, "Petrol", "VXi"))
print(f"Maruti Swift 2015, 45,000 km, petrol → ₹{p.predicted_price_inr:,}  (90% interval ₹{p.lower_inr:,} – ₹{p.upper_inr:,})")
display(pd.DataFrame(p.factors))

## Step 9 · Monitoring demo (data drift vs. real performance)
Uses the held-out listings as stand-in 'traffic'. Real use: pass recent listings, and verified sale prices when known.

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
import monitor
from carprice.models.bundle import latest_bundle_dir
traffic = pd.read_csv(P.data_processed / "test.csv")
rep = monitor.run(traffic[["name","company","year","kms_driven","fuel_type"]], latest_bundle_dir(P.models), settings.cfg["monitoring"], P.reports / "monitoring")
print("drift:", rep["drift"]["overall_drift_status"], "| performance:", rep["performance"]["status"], "| decision:", rep["decision"]["action"])
print(rep["performance"].get("reason", ""))

## Step 10 · (Optional) Interactive app
Set `LAUNCH_APP = True` above. Gradio creates a public `*.gradio.live` link while the cell runs.

In [ ]:
if LAUNCH_APP:
    os.environ["CARPRICE_MODELS_DIR"] = str(P.models)
    from app.app import build_app
    build_app().launch(share=True)
else:
    print("Skipped. REST API (FastAPI) locally:  CARPRICE_API_KEY=... uvicorn api.main:app   |   Docker: see README.")

## Step 11 · Persist everything to Drive and package a download

In [ ]:
mlruns = Path(os.environ.get("CARPRICE_MLFLOW_DIR", P.artifacts / "mlruns"))
if mlruns.exists() and mlruns.resolve() != (P.artifacts / "mlruns").resolve():
    shutil.copytree(mlruns, P.artifacts / "mlruns", dirs_exist_ok=True)
    print("MLflow runs copied to Drive")
archive = shutil.make_archive(str(HOME / "car_price_results"), "zip", root_dir=HOME, base_dir="reports")
print("Results archive:", archive)
print("\nEverything lives under:", HOME)
for sub in ["data/raw", "data/interim", "data/processed", "artifacts/models", "artifacts/checkpoints", "artifacts/logs", "reports/figures", "reports/metrics", "reports/model_reports"]:
    print(f"  {sub:26s}", "exists" if (HOME / sub).exists() else "-")
if IN_COLAB:
    from google.colab import files
    files.download(archive)
status()